## Import

In [ ]:
import pandas as pd
import json

import re

import importlib
import pipeline.utils

importlib.reload(pipeline.utils)

from pipeline.utils import score_data_quality, summarize_quality

In [73]:
df = pd.read_csv('data/minimum/minimum_cleaning.csv.zip')

In [74]:
def normalize(col):
    return re.sub(r'[^a-z]', '', col.lower())

## Remove Duplicates

In [75]:
def remove_exact_duplicates(df):
    before = len(df)
    df_clean = df.drop_duplicates()
    after = len(df_clean)

    return df_clean, {
        "type": "exact",
        "removed_rows": before - after
    }


# --- Detect ID-like column ---
def detect_id_column(df):
    for col in df.columns:
        norm = normalize(col)

        # safer ID detection
        if norm == "id" or norm.endswith("id"):
            return col

    return None


# --- ID-based deduplication ---
def remove_id_duplicates(df, id_col):
    before = len(df)

    df_clean = df.drop_duplicates(subset=[id_col], keep="first")

    after = len(df_clean)

    return df_clean, {
        "type": "id_based",
        "column": id_col,
        "removed_rows": before - after
    }


# --- Auto-detect content columns ---
def detect_content_columns(df):
    content_cols = []

    for col in df.columns:
        norm = normalize(col)

        # skip obvious non-content columns
        if (
            "id" in norm or
            "date" in norm or
            "time" in norm
        ):
            continue

        # keep only useful columns
        if df[col].nunique() > 1:
            content_cols.append(col)

    return content_cols


# --- Content-based deduplication ---
def remove_content_duplicates(df, subset_cols):
    if len(subset_cols) < 2:
        return df, {
            "type": "content_based",
            "columns": subset_cols,
            "removed_rows": 0
        }

    before = len(df)

    df_clean = df.drop_duplicates(subset=subset_cols, keep="first")

    after = len(df_clean)

    return df_clean, {
        "type": "content_based",
        "columns": subset_cols,
        "removed_rows": before - after
    }


# --- Main handler ---
def handle_duplicates(df):
    df = df.copy()
    report = []

    # 1. Exact duplicates
    df, report_exact = remove_exact_duplicates(df)
    report.append(report_exact)

    # 2. ID-based duplicates (if exists)
    id_col = detect_id_column(df)
    if id_col:
        df, report_id = remove_id_duplicates(df, id_col)
        report.append(report_id)

    # 3. Content-based duplicates
    content_cols = detect_content_columns(df)
    df, report_content = remove_content_duplicates(df, content_cols)
    report.append(report_content)

    return df, report

## Value Validation

In [76]:
def validate_inputs(df):
    df = df.copy()
    invalid_mask = pd.Series(False, index=df.index)

    for col in df.columns:

        # --- Numeric validation ---
        if df[col].dtype == "object":
            try:
                is_numeric = df[col].str.match(r"^\d+(\.\d+)?$", na=False)
                if is_numeric.any():
                    invalid_mask |= ~df[col].str.match(r"^\d+(\.\d+)?$", na=True)
            except:
                pass

        # --- Age range ---
        if col.lower() == "age":
            invalid_mask |= (df[col] < 0) | (df[col] > 120)

    valid_df = df[~invalid_mask]
    invalid_df = df[invalid_mask]

    report = {
        "invalid_rows": len(invalid_df)
    }

    return valid_df, invalid_df, report

## Missing Value

In [77]:
def handle_missing(df):
    df = df.copy()

    missing_mask = df.isnull().any(axis=1)

    handled_df = df[~missing_mask]
    unhandled_df = df[missing_mask]

    report = {
        "rows_with_missing": len(unhandled_df)
    }

    return handled_df, unhandled_df, report

## Type Conversion

In [78]:
def is_date_column(col_name):
    norm = normalize(col_name)

    date_keywords = ["date", "time", "dob", "birth", "day"]

    return any(keyword in norm for keyword in date_keywords)


def try_parse_dates(series):
    try:
        parsed = pd.to_datetime(series, errors='coerce')
        return parsed
    except:
        return None
    
def looks_like_date(series):
    sample = series.dropna().astype(str).head(5)

    try:
        pd.to_datetime(sample, errors='coerce')
        return True
    except:
        return False

In [79]:
def convert_types(df):
    df = df.copy()
    conversion_report = []

    for col in df.columns:
        if df[col].dtype != "object":
            continue

        series = df[col].dropna().astype(str)

        # --- 1. Check if numeric-like (light check) ---
        numeric_ratio = series.str.match(r"^\d+(\.\d+)?$").mean()

        if numeric_ratio >= 0.8:  # only convert if confident
            df[col] = pd.to_numeric(df[col], errors='coerce')
            conversion_report.append({
                "column": col,
                "type": "numeric"
            })
            continue

        # --- 2. Date conversion (only if likely) ---
        if is_date_column(col):
            parsed = pd.to_datetime(df[col], errors='coerce')

            success_ratio = parsed.notna().mean()

            if success_ratio >= 0.8:
                df[col] = parsed
                conversion_report.append({
                    "column": col,
                    "type": "datetime"
                })

    return df, conversion_report

## Value Normalization

In [80]:
def normalize_gender(val):
    if pd.isna(val):
        return val

    v = str(val).strip().lower()

    male_set = {"m", "male", "man"}
    female_set = {"f", "female", "woman"}

    if v in male_set:
        return "Male"
    elif v in female_set:
        return "Female"

    return "Other"


def normalize_boolean(val):
    if pd.isna(val):
        return val

    v = str(val).strip().lower()

    if v in {"yes", "true", "1"}:
        return True
    elif v in {"no", "false", "0"}:
        return False

    return val


def normalize_text(val):
    if pd.isna(val):
        return val

    return str(val).strip().title()

In [81]:
def normalize_values(df):
    df = df.copy()

    # --- Gender ---
    if "Gender" in df.columns:
        df["Gender"] = df["Gender"].apply(normalize_gender)

    # --- Boolean normalization ---
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(normalize_boolean)

    # --- Light text normalization (safe only) ---
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(lambda x: normalize_text(x) if isinstance(x, str) else x)

    return df

## Scoring Data Quality

In [82]:
def run_step2(df):
    reports = {}

    # duplicates
    df, duplicate_report = handle_duplicates(df)
    reports["duplicates"] = duplicate_report

    # validation
    df_valid, df_invalid, validation_report = validate_inputs(df)
    reports["validation"] = validation_report

    # missing
    df_clean, df_missing, missing_report = handle_missing(df_valid)
    reports["missing"] = missing_report

    # type conversion
    df_clean, conversion_report = convert_types(df_clean)
    reports["type_conversion"] = conversion_report

    # normalization
    df_clean = normalize_values(df_clean)

    # scoring again (optional)
    df_clean = score_data_quality(df_clean)
    reports["quality"] = summarize_quality(df_clean)

    return df_clean, df_invalid, df_missing, reports

In [83]:
df_clean, df_invalid, df_missing, reports = run_step2(df)

## Export

In [84]:
df_clean.to_csv('data/full/full_cleaning.csv.zip', index=False, compression='zip')

with open('data/full/full_cleaning_report.json', 'w') as f:
    json.dump(reports, f, indent=2)